In [4]:
from docx import Document
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

# Load the document
doc = Document("data.docx")

# Get the first table
table = doc.tables[0]

# Extract the table into a list of lists
data = []
for row in table.rows:
    data.append([cell.text.strip() for cell in row.cells])

# Convert to DataFrame
df = pd.DataFrame(data[1:], columns=data[0])

# Clean up column names (strip spaces, replace weird characters)
df.columns = [col.strip().replace("", "Δ") for col in df.columns]

# Check exact column names
print("Columns:", df.columns.tolist())

# Filter rows where Type == "CB"
df_cb = df[df["Type"] == "CB"]

# Convert relevant columns to numeric
df_cb["Hs/ΔD"] = pd.to_numeric(df_cb["Hs/ΔD"], errors="coerce")
df_cb["N"] = pd.to_numeric(df_cb["N"], errors="coerce")

# Drop rows with missing numeric data
df_cb = df_cb.dropna(subset=["Hs/ΔD", "N", "Updated damage"])

# Define color mapping
damage_colors = {
    "0": "white",
    "a": "green",
    "b": "yellow",
    "c": "orange",
    "d": "red",
    "c1": "orange",
    "d1": "red",
}

# Split DataFrame into:
# → all CB rows except c1, d1
df_circles = df_cb[~df_cb["Updated damage"].isin(["c1", "d1"])]

# → CB rows with c1 or d1
df_crosses = df_cb[df_cb["Updated damage"].isin(["c1", "d1"])]

# ➡️ Subset for second plot:
df_cb_rect = df_cb[df_cb["Toplayer"] == "Rectangular blocks"]

# Split second subset into circles and crosses
df_circles_rect = df_cb_rect[~df_cb_rect["Updated damage"].isin(["c1", "d1"])]
df_crosses_rect = df_cb_rect[df_cb_rect["Updated damage"].isin(["c1", "d1"])]

# Plot both plots as subplots
fig, axes = plt.subplots(2, 1, figsize=(10, 12), sharex=True)

# --- First subplot: all CB
ax = axes[0]

# Circles
ax.scatter(
    df_circles["N"],
    df_circles["Hs/ΔD"],
    c=df_circles["Updated damage"].map(damage_colors),
    edgecolor='black',
    s=80,
    marker='o'
)

# Crosses
ax.scatter(
    df_crosses["N"],
    df_crosses["Hs/ΔD"],
    c=df_crosses["Updated damage"].map(damage_colors),
    edgecolor='black',
    s=120,
    marker='x',
    linewidth=2
)

ax.set_xlim(0, 2000)
ax.set_title("Hs/ΔD vs N for Type CB (All)", fontsize=16)
ax.set_ylabel("Hs/ΔD", fontsize=14)

legend_elements = [
    Patch(facecolor="white", edgecolor='black', label='0 - No damage'),
    Patch(facecolor="green", edgecolor='black', label='a - Start/revetment damage'),
    Patch(facecolor="yellow", edgecolor='black', label='b - Revetment damage'),
    Patch(facecolor="orange", edgecolor='black', label='c / c1 - Start revetment failure'),
    Patch(facecolor="red", edgecolor='black', label='d / d1 - Revetment failure'),
]
ax.legend(handles=legend_elements, title="Updated Damage", fontsize=12, title_fontsize=13)

ax.tick_params(axis='x', labelrotation=45, labelsize=12)
ax.tick_params(axis='y', labelsize=12)
ax.grid(True, linestyle='--', alpha=0.5)

# --- Second subplot: CB with Rectangular blocks
ax2 = axes[1]

# Circles
ax2.scatter(
    df_circles_rect["N"],
    df_circles_rect["Hs/ΔD"],
    c=df_circles_rect["Updated damage"].map(damage_colors),
    edgecolor='black',
    s=80,
    marker='o'
)

# Crosses
ax2.scatter(
    df_crosses_rect["N"],
    df_crosses_rect["Hs/ΔD"],
    c=df_crosses_rect["Updated damage"].map(damage_colors),
    edgecolor='black',
    s=120,
    marker='x',
    linewidth=2
)

ax2.set_xlim(0, 2000)
ax2.set_xlabel("N", fontsize=14)
ax2.set_ylabel("Hs/ΔD", fontsize=14)
ax2.set_title("Hs/ΔD vs N for Type CB with Rectangular blocks", fontsize=16)

ax2.tick_params(axis='x', labelrotation=45, labelsize=12)
ax2.tick_params(axis='y', labelsize=12)
ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

# Optional: print filtered dataframe
print(df_cb_rect)


ModuleNotFoundError: No module named 'docx'

In [ ]:
from docx import Document
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.legend_handler import HandlerTuple

# Load the document
doc = Document("data.docx")

# Get the first table
table = doc.tables[0]

# Extract the table into a list of lists
data = []
for row in table.rows:
    data.append([cell.text.strip() for cell in row.cells])

# Convert to DataFrame
df = pd.DataFrame(data[1:], columns=data[0])

# Clean up column names (strip spaces, replace weird characters)
df.columns = [col.strip().replace("", "Δ") for col in df.columns]

# Check exact column names
print("Columns:", df.columns.tolist())

# Filter rows where Type == "CB" and Toplayer == "Rectangular blocks"
df_cb_rect = df[
    (df["Type"] == "CB") &
    (df["Toplayer"] == "Rectangular blocks")
]

# Convert relevant columns to numeric
df_cb_rect["Hs/ΔD"] = pd.to_numeric(df_cb_rect["Hs/ΔD"], errors="coerce")
df_cb_rect["N"] = pd.to_numeric(df_cb_rect["N"], errors="coerce")

# Drop rows with missing numeric data
df_cb_rect = df_cb_rect.dropna(subset=["Hs/ΔD", "N", "Updated damage"])

# Define color mapping
damage_colors = {
    "0": "white",
    "a": "green",
    "b": "yellow",
    "c": "orange",
    "d": "red",
    "c1": "orange",
    "d1": "red",
}

# Split into circles and crosses
df_circles = df_cb_rect[~df_cb_rect["Updated damage"].isin(["c1", "d1"])]
df_crosses = df_cb_rect[df_cb_rect["Updated damage"].isin(["c1", "d1"])]

# --- Add your own data point ---
my_N = 1000
my_damage = "d"

# Define densities
rho_s = 2300     # block density (kg/m³)
rho_w = 1000     # water density (kg/m³)

# Calculate Δ
Delta = (rho_s - rho_w) / rho_w

# Calculate Hs/(Δ·D)
Hs = 0.7
D = 0.15
my_Hs_Delta_D = Hs / (Delta * D)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

# Circles
ax.scatter(
    df_circles["N"],
    df_circles["Hs/ΔD"],
    c=df_circles["Updated damage"].map(damage_colors),
    edgecolor='black',
    s=80,
    marker='o'
)

# Crosses
ax.scatter(
    df_crosses["N"],
    df_crosses["Hs/ΔD"],
    c=df_crosses["Updated damage"].map(damage_colors),
    edgecolor='black',
    s=80,
    marker='+',
    linewidth=2
)

# Plot your own data point
ax.scatter(
    my_N,
    my_Hs_Delta_D,
    color=damage_colors[my_damage],
    edgecolor='black',
    s=80,
    marker='s',
    linewidth=1.5
)

# Labels and title
ax.set_xlim(0, 2000)
ax.set_xlabel("N", fontsize=14)
ax.set_ylabel("Hs/ΔD", fontsize=14)
ax.set_title("Hs/ΔD vs N for Type CB with Rectangular blocks", fontsize=16)

# Remove legend entirely (no ax.legend call)

# Ticks formatting
ax.tick_params(axis='x', labelrotation=45, labelsize=12)
ax.tick_params(axis='y', labelsize=12)

# Grid
ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

# Optional: print filtered dataframe
print(df_cb_rect)


In [ ]:
from docx import Document
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.legend_handler import HandlerTuple

# Load the document
doc = Document("data.docx")

# Get the first table
table = doc.tables[0]

# Extract the table into a list of lists
data = []
for row in table.rows:
    data.append([cell.text.strip() for cell in row.cells])

# Convert to DataFrame
df = pd.DataFrame(data[1:], columns=data[0])

# Clean up column names (strip spaces, replace weird characters)
df.columns = [col.strip().replace("", "Δ") for col in df.columns]

# Check exact column names
print("Columns:", df.columns.tolist())

# Filter rows where Type == "CB" and Toplayer == "Rectangular blocks"
df_cb_rect = df[
    (df["Type"] == "CB") &
    (df["Toplayer"] == "Rectangular blocks")
]

# Convert relevant columns to numeric
df_cb_rect["Hs/ΔD"] = pd.to_numeric(df_cb_rect["Hs/ΔD"], errors="coerce")
df_cb_rect["N"] = pd.to_numeric(df_cb_rect["N"], errors="coerce")

# Drop rows with missing numeric data
df_cb_rect = df_cb_rect.dropna(subset=["Hs/ΔD", "N", "Updated damage"])

# Define color mapping
damage_colors = {
    "0": "white",
    "a": "green",
    "b": "yellow",
    "c": "orange",
    "d": "red",
    "c1": "orange",
    "d1": "red",
}

# Split into circles and crosses
df_circles = df_cb_rect[~df_cb_rect["Updated damage"].isin(["c1", "d1"])]
df_crosses = df_cb_rect[df_cb_rect["Updated damage"].isin(["c1", "d1"])]

# --- Add your own data point ---
my_N = 1000
my_damage = "d"

# Define densities
rho_s = 2300     # block density (kg/m³)
rho_w = 1000     # water density (kg/m³)

# Calculate Δ
Delta = (rho_s - rho_w) / rho_w

# Calculate Hs/(Δ·D)
Hs = 0.7
D = 0.15
my_Hs_Delta_D = Hs / (Delta * D)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

# Circles
ax.scatter(
    df_circles["N"],
    df_circles["Hs/ΔD"],
    c=df_circles["Updated damage"].map(damage_colors),
    edgecolor='black',
    s=80,
    marker='o'
)

# Crosses
ax.scatter(
    df_crosses["N"],
    df_crosses["Hs/ΔD"],
    c=df_crosses["Updated damage"].map(damage_colors),
    edgecolor='black',
    s=80,
    marker='+',
    linewidth=2
)

# Plot your own data point
ax.scatter(
    my_N,
    my_Hs_Delta_D,
    color=damage_colors[my_damage],
    edgecolor='black',
    s=80,
    marker='s',
    linewidth=1.5,
    label='My Data Point'
)

# Labels and title
ax.set_xlim(0, 2000)
ax.set_xlabel("N", fontsize=14)
ax.set_ylabel("Hs/ΔD", fontsize=14)
ax.set_title("Hs/ΔD vs N for Type CB with Rectangular blocks", fontsize=16)

legend_elements = [
    Line2D([0], [0],
           marker='o',
           color='white',
           markerfacecolor='white',
           markeredgecolor='black',
           markersize=10,
           linestyle='None',
           label='0 - No damage'),

    Line2D([0], [0],
           marker='o',
           color='green',
           markerfacecolor='green',
           markeredgecolor='black',
           markersize=10,
           linestyle='None',
           label='a - Start/revetment damage'),

    Line2D([0], [0],
           marker='o',
           color='yellow',
           markerfacecolor='yellow',
           markeredgecolor='black',
           markersize=10,
           linestyle='None',
           label='b - Revetment damage'),

    Line2D([0], [0],
           marker='o',
           color='orange',
           markerfacecolor='orange',
           markeredgecolor='black',
           markersize=10,
           linestyle='None',
           label='b - Revetment damage'),

    Line2D([0], [0],
           marker='o',
           color='yellow',
           markerfacecolor='yellow',
           markeredgecolor='black',
           markersize=10,
           linestyle='None',
           label='b - Revetment damage'),

    Line2D([0], [0],
           marker='+',
           color='orange',
           markerfacecolor='orange',
           markeredgecolor='black',
           markersize=10,   # plus symbols need slightly larger size to be visible
           linestyle='None',
           label='c / c1 - Start revetment failure'),

    Line2D([0], [0],
           marker='+',
           color='red',
           markerfacecolor='red',
           markeredgecolor='black',
           markersize=14,
           linestyle='None',
           label='d / d1 - Revetment failure'),

    Line2D([0], [0],
           marker='s',
           color='red',
           markerfacecolor='red',
           markeredgecolor='black',
           markersize=10,
           linestyle='None',
           label='My Data Point'),
]

ax.legend(handles=legend_elements, title="Updated Damage", fontsize=12, title_fontsize=13)
# Ticks formatting
ax.tick_params(axis='x', labelrotation=45, labelsize=12)
ax.tick_params(axis='y', labelsize=12)

# Grid
ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

# Optional: print filtered dataframe
print(df_cb_rect)
